# L16 · Failure Diagnosis, Evaluation, and Capstone

## Goal

- detect reward hacking and collapse in metrics
- audit fair-comparison conditions
- separate local results from paper results

## Setup

This cell fixes CPU, seed, offline status, and the split hash first. Toy code uses deterministic CPU operations; package trainers retain their strict global default.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L16:toy:42").hexdigest()
print(f"lesson=L16 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L16 language=en profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.12.13 rl_study=0.1.0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:57e71e09fbafbac3d140c89d9b3a3be786a33338429f945829a582cff0c91d83 data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. Position and core equation

⏱ 5 min · 1/3 section · [CORE]

Position: every training path → **evaluation, failure diagnosis, and reproducibility audit**

$$\text{fair comparison}=\text{same init}+\text{same data}+\text{same budget}+\text{same metric contract}$$

One good score can hide reward hacking, length bias, or entropy collapse. Fair comparisons require matching initial hashes, data order, token/forward/environment budgets, and metric definitions. Local toy results and paper benchmarks must not be ranked in one table as if conditions matched.

### 2. Run with small numbers

⏱ 6 min · 2/3 section · [CORE]

**Predict first:** Even if three report files look successful, can they count as local evidence without `result_origin`? Write an answer for 20 seconds, then run the cell.

<details><summary>Show answer</summary>No. Without origin, environment, config, and budget, executed results cannot be distinguished from example data.</details>

In [2]:
report_paths = [
    ROOT / "docs/research/C6_ALIGNMENT_BENCHMARK.json",
    ROOT / "docs/research/C7_GROUP_BENCHMARK.json",
    ROOT / "docs/research/C9_AGENTIC_BENCHMARK.json",
]
audit_rows = []
for report_path in report_paths:
    payload = json.loads(report_path.read_text(encoding="utf-8"))
    audit_rows.append({
        "report": report_path.name,
        "origin": payload.get("result_origin"),
        "has_sources": bool(payload.get("sources")),
        "has_guardrail": bool(payload.get("interpretation") or payload.get("interpretation_guardrails")),
    })
print(audit_rows)

[{'report': 'C6_ALIGNMENT_BENCHMARK.json', 'origin': 'local_executed', 'has_sources': False, 'has_guardrail': True}, {'report': 'C7_GROUP_BENCHMARK.json', 'origin': 'local_executed', 'has_sources': True, 'has_guardrail': True}, {'report': 'C9_AGENTIC_BENCHMARK.json', 'origin': 'local_executed', 'has_sources': True, 'has_guardrail': True}]


### 3. Implementation anatomy

⏱ 6 min · 3/3 section · [DEEP DIVE]

**Why this implementation:** The capstone machine-reads contracts from existing artifacts instead of starting another large run. More seeds reduce statistical uncertainty but do not repair mismatched comparison conditions.

**Common trap:** Comparing equal `steps` under different token budgets favors algorithms with longer responses. Report multiple budget counters and split hashes together. Regression tests: `test_alignment_comparison_audits_shared_start_and_prompts`.

**Checkpoint:** Continue when you can explain just one printed value.

## Checks

In [3]:
assert len(audit_rows) == 3
assert all(row["origin"] == "local_executed" for row in audit_rows)
print("checks=passed")

checks=passed


**Recall:** If exact match is high and entropy is near zero, which two interpretations must you distinguish next? Answer in one or two sentences.

## Mistakes I Revisit

- Assuming a finite loss proves the implementation is correct.
- Merging `terminated` with `truncated`, or prompt with action.
- Turning one tiny seed into an algorithm ranking.

## 60-Second Recap

- **Run conclusion:** C6, C7, and C9 reports all declare `local_executed` and include interpretation guardrails. The audit also exposes a gap: C6 lacks an embedded `sources` array and needs stronger source traceability.
- Executable checks: `test_alignment_comparison_audits_shared_start_and_prompts`.
- The output is a fixed-seed toy run, not a paper-scale result.

## Next Steps

1. Return to the README fast/full routes, revisit weak areas, and design your own experiment with the reproducibility checklist.
2. Break one `[CORE]` assertion and read the failure.
3. Open the package test and connect the notebook equation to its production guard.

[Implementation note](../../docs/research/README.md) · [Course map](../../docs/course-map.en.md)

## Sources

- `sutton-barto-rl2` — `docs/sources.yml`
- `ppo-2017` — `docs/sources.yml`
- `dpo-2023` — `docs/sources.yml`
- `deepseekmath-grpo-2024` — `docs/sources.yml`
- `dapo-2025` — `docs/sources.yml`
- `agent-lightning-2025` — `docs/sources.yml`